In [1]:
!pip install rdkit pubchempy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 38.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import pubchempy as pcp

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import AllChem
from rdkit.DataStructs import TanimotoSimilarity

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

In [6]:
!wget -O delaney-processed.csv https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv

--2026-06-11 12:59:15--  https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/delaney-processed.csv
Resolving deepchemdata.s3-us-west-1.amazonaws.com (deepchemdata.s3-us-west-1.amazonaws.com)... 52.219.220.122, 16.15.0.18, 16.15.0.53, ...
Connecting to deepchemdata.s3-us-west-1.amazonaws.com (deepchemdata.s3-us-west-1.amazonaws.com)|52.219.220.122|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 96699 (94K) [text/csv]
Saving to: ‘delaney-processed.csv’

delaney-processed.c 100%[===================>]  94.43K  --.-KB/s    in 0.1s    

2026-06-11 12:59:15 (782 KB/s) - ‘delaney-processed.csv’ saved [96699/96699]



In [7]:
import pandas as pd

df = pd.read_csv("delaney-processed.csv")

print("Total Molecules:", len(df))
print("Columns:")
print(df.columns.tolist())

Total Molecules: 1128
Columns:
['Compound ID', 'ESOL predicted log solubility in mols per litre', 'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors', 'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area', 'measured log solubility in mols per litre', 'smiles']


In [8]:
print("Dataset Shape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (1128, 10)

Missing Values:
Compound ID                                        0
ESOL predicted log solubility in mols per litre    0
Minimum Degree                                     0
Molecular Weight                                   0
Number of H-Bond Donors                            0
Number of Rings                                    0
Number of Rotatable Bonds                          0
Polar Surface Area                                 0
measured log solubility in mols per litre          0
smiles                                             0
dtype: int64

Duplicate Rows:
0


In [9]:
from rdkit import Chem

valid_smiles = 0
invalid_smiles = 0

for smi in df["smiles"]:
    mol = Chem.MolFromSmiles(str(smi))

    if mol:
        valid_smiles += 1
    else:
        invalid_smiles += 1

print("Valid SMILES:", valid_smiles)
print("Invalid SMILES:", invalid_smiles)

Valid SMILES: 1128
Invalid SMILES: 0


In [10]:
from rdkit import Chem

df["Mol"] = df["smiles"].apply(Chem.MolFromSmiles)

print("Molecule Objects Created:", df["Mol"].notnull().sum())

Molecule Objects Created: 1128


In [11]:
from rdkit.Chem import Descriptors

def get_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticRings": Descriptors.NumAromaticRings(mol)
    }

In [13]:
descriptor_list = []

for mol in df["Mol"]:
    descriptor_list.append(get_descriptors(mol))

descriptor_df = pd.DataFrame(descriptor_list)

print(descriptor_df.shape)
descriptor_df

(1128, 7)


,MolWt,LogP,TPSA,HBD,HBA,RotBonds,AromaticRings
0,457.432,-3.10802,202.32,7,12,7,1
1,201.225,2.84032,42.24,1,2,2,2
2,152.237,2.87800,17.07,0,1,4,0
3,278.354,6.29940,0.00,0,0,0,5
4,84.143,1.74810,0.00,0,1,0,1
...,...,...,...,...,...,...,...
1123,197.381,2.50850,0.00,0,0,0,0
1124,219.266,0.10710,71.00,1,5,1,0
1125,246.359,2.99000,18.46,0,5,7,0
1126,72.151,2.05240,0.00,0,0,1,0


In [14]:
df_final = pd.concat([df, descriptor_df], axis=1)

print(df_final.shape)
df_final.columns

(1128, 18)


Index(['Compound ID', 'ESOL predicted log solubility in mols per litre',
       'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors',
       'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area',
       'measured log solubility in mols per litre', 'smiles', 'Mol', 'MolWt',
       'LogP', 'TPSA', 'HBD', 'HBA', 'RotBonds', 'AromaticRings'],
      dtype='object')

In [15]:
from rdkit.Chem import Lipinski

def lipinski_filter(mol):
    return {
        "MW_OK": Descriptors.MolWt(mol) <= 500,
        "LogP_OK": Descriptors.MolLogP(mol) <= 5,
        "HBD_OK": Lipinski.NumHDonors(mol) <= 5,
        "HBA_OK": Lipinski.NumHAcceptors(mol) <= 10
    }

lipinski_results = []

for mol in df["Mol"]:
    lipinski_results.append(lipinski_filter(mol))

lipinski_df = pd.DataFrame(lipinski_results)

df_final = pd.concat([df_final, lipinski_df], axis=1)

print(df_final.shape)

(1128, 22)


In [16]:
df_final["DrugLikeScore"] = (
    df_final["MW_OK"].astype(int) +
    df_final["LogP_OK"].astype(int) +
    df_final["HBD_OK"].astype(int) +
    df_final["HBA_OK"].astype(int)
)

In [17]:
df_final["DrugLikeScore"].value_counts().sort_index()

,count
DrugLikeScore,
1,2
2,10
3,98
4,1018


In [18]:
drug_like = df_final[df_final["DrugLikeScore"] == 4]

print("Total Drug-like molecules:", len(drug_like))

Total Drug-like molecules: 1018


In [19]:
df_final = df_final.sort_values(by="DrugLikeScore", ascending=False)

df_final.to_csv("project15_drug_discovery_output.csv", index=False)

In [20]:
features = [
    "MolWt", "LogP", "TPSA",
    "HBD", "HBA",
    "RotBonds", "AromaticRings"
]

X = df_final[features]
y = df_final["measured log solubility in mols per litre"]

print(X.shape, y.shape)

(1128, 7) (1128,)


In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, random_state=42)

In [23]:
y_pred = model.predict(X_test)

In [25]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

r2 = r2_score(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("R² Score:", r2)
print("RMSE:", rmse)

R² Score: 0.8816637447489515
RMSE: 0.7033173061167072


In [26]:
from rdkit.Chem import AllChem
from rdkit import DataStructs

def get_fingerprint(mol):
    return AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)

In [28]:
df_final["Fingerprint"] = df_final["Mol"].apply(get_fingerprint)

[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerator
[13:09:35] DEPRECATION WARNING: please use MorganGenerat

In [29]:
from rdkit import DataStructs

query_mol = df_final[df_final["DrugLikeScore"] == 4]["Mol"].iloc[0]
query_fp = get_fingerprint(query_mol)

similarities = []

for fp in df_final["Fingerprint"]:
    sim = DataStructs.TanimotoSimilarity(query_fp, fp)
    similarities.append(sim)

df_final["Similarity"] = similarities

df_final[["Compound ID", "Similarity", "DrugLikeScore"]].sort_values(by="Similarity", ascending=False).head(10)

[13:10:01] DEPRECATION WARNING: please use MorganGenerator


,Compound ID,Similarity,DrugLikeScore
1126,2-Methylbutane,1.000000,4
180,3-Methylpentane,0.545455,4
193,2-Chlorobutane,0.500000,4
842,Butan-2-ol,0.500000,4
806,"2,3-Dimethylpentane",0.461538,4
478,2-Methylpentane,0.461538,4
335,"2,4-Dimethylpentane",0.454545,4
474,1-Bromo-2-methylpropane,0.416667,4
815,1-Chloro-2-methylpropane,0.416667,4
1073,2-Methylpropan-1-ol,0.416667,4


In [30]:
df_final["PredictedSol_norm"] = (
    df_final["ESOL predicted log solubility in mols per litre"] -
    df_final["ESOL predicted log solubility in mols per litre"].min()
) / (
    df_final["ESOL predicted log solubility in mols per litre"].max() -
    df_final["ESOL predicted log solubility in mols per litre"].min()
)

In [31]:
df_final["Similarity_norm"] = df_final["Similarity"]

In [32]:
df_final["FinalScore"] = (
    0.4 * df_final["DrugLikeScore"] +
    0.3 * df_final["PredictedSol_norm"] +
    0.3 * df_final["Similarity_norm"]
)

In [33]:
final_ranked = df_final.sort_values(by="FinalScore", ascending=False)

final_ranked[[
    "Compound ID",
    "FinalScore",
    "DrugLikeScore",
    "Similarity"
]].head(15)

,Compound ID,FinalScore,DrugLikeScore,Similarity
1126,2-Methylbutane,2.107273,4,1.000000
842,Butan-2-ol,2.002553,4,0.500000
1073,2-Methylpropan-1-ol,1.975996,4,0.416667
193,2-Chlorobutane,1.965751,4,0.500000
16,2-Methylbutanol,1.961129,4,0.400000
180,3-Methylpentane,1.961042,4,0.545455
992,3-Pentanol,1.958097,4,0.384615
747,2-Methyl-3-pentanol,1.953318,4,0.400000
983,Ethanol,1.952049,4,0.272727
651,"1,1-Diethoxyethane",1.944686,4,0.333333
